# TS2Vec Minimal Structural-Break Showcase

This notebook is reduced to the minimal self-supervised showcase:

1. train TS2Vec for 15 epochs on the left-of-boundary segments
2. score structural breaks via cosine mismatch between left and right embeddings
3. render slide-friendly diagnostics

The old classifier, LightGBM, and scratch exploration cells were removed so the run order is linear.

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

work_dir = "/kaggle/input/adia-break-prediction"
y_train = pd.read_parquet(os.path.join(work_dir, "y_train.parquet"))
X_train = pd.read_parquet(os.path.join(work_dir, "x_train.parquet"))

y_target = np.asarray(y_train).reshape(-1)
print(X_train.shape, y_train.shape)

In [ ]:
import torch
import numpy as np
from ts2vec.utils import take_per_row

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


class CroppingDataset(Dataset):
    def __init__(self, data_list, seq_length=128, temporal_unit=2, diff_proba: float = 0, resample_proba: float = 0, normalize_proba: float = 0):
        """
        Args:
            data_list: list of np.ndarray, каждый [T_i, D]
            seq_length: длина последовательности на выходе из __getitem__
            temporal_unit: минимальный лог2-длины base crop
        """
        self.data = data_list
        self.seq_length = seq_length
        self.temporal_unit = temporal_unit
        self.diff_proba = diff_proba
        self.resample_proba = resample_proba
        self.normalize_proba = normalize_proba

        
    def __len__(self):
        return len(self.data)

    def _crop(self, x):
        T = x.shape[0]
        if T > self.seq_length:
            start = np.random.randint(T - self.seq_length + 1)
            x = x[start : start + self.seq_length]
        elif T < self.seq_length:
            pad_len = self.seq_length - T
            x = np.pad(x, ((0, pad_len), (0, 0)), mode="constant")
        return x

    def resample_mean_pad(self, x, factor):
        T = x.shape[0]
        if T < self.seq_length * factor:
            return T
        remainder = T % factor
        if remainder != 0:
            pad_len = factor - remainder
            pad = np.repeat(x[-1:], pad_len, axis=0)
            x = np.concatenate([x, pad], axis=0)
        return x.reshape(-1, factor, *x.shape[1:]).mean(axis=1)

    @staticmethod
    def normalize(x, eps=1e-9):
        x = (x - x.mean()) / (x.std() + 1e-9)
        return x
        
    def __getitem__(self, idx):
        """сначала сэмплируем последовательность длины seq len"""
        x = self.data[idx]
        
        if np.random.rand() < self.diff_proba:
            x = np.diff(x, axis=0)
        if np.random.rand() < self.resample_proba:
            x = self.resample_mean_pad(x, factor=2)
        x = self._crop(x)
        if np.random.rand() < self.normalize_proba:
            x = self.normalize(x)
        return x

    def collate_fn(self, batch):
        """
        batch: list of [seq_length, D] (np.ndarrays) → stack → [B, seq_length, D]
        применяет кроп на уровне батча
        """
        x = torch.tensor(batch, dtype=torch.float32)  # [B, T, D]
        seq_length = x.size(1)
        batch_size = x.size(0)

        base_crop_length = np.random.randint(low=2 ** (self.temporal_unit + 1), high=seq_length + 1)
        base_crop_left_idx = np.random.randint(seq_length - base_crop_length + 1)
        base_crop_right_idx = base_crop_left_idx + base_crop_length
        extended_crop_left_idx = np.random.randint(base_crop_left_idx + 1)
        extended_crop_right_idx = np.random.randint(low=base_crop_right_idx, high=seq_length + 1)

        crop_offset = np.random.randint(
            low=-extended_crop_left_idx,
            high=seq_length - extended_crop_right_idx + 1,
            size=batch_size,
        )

        first_crop = take_per_row(x, crop_offset + extended_crop_left_idx, base_crop_right_idx - extended_crop_left_idx)
        second_crop = take_per_row(x, crop_offset + base_crop_left_idx, extended_crop_right_idx - base_crop_left_idx)

        return first_crop, second_crop, base_crop_length

In [ ]:
def prepare_positive_train_data(X_train, y_train, left_length, right_length, sample_idx=None):
    series_list = []
    desired_len = abs(left_length) + abs(right_length)

    sample_ids = y_train.index if sample_idx is None else y_train.iloc[sample_idx].index

    for ts_id in tqdm(sample_ids):
        ts = X_train.loc[ts_id]["value"].values.astype(np.float32)
        boundary = (X_train.loc[ts_id].period == 0).sum()

        start_idx = boundary + left_length
        end_idx = boundary + right_length

        if start_idx < 0:
            continue

        ts_slice = ts[start_idx:end_idx]
        if len(ts_slice) < desired_len:
            pad = np.full((desired_len - len(ts_slice),), np.nan, dtype=np.float32)
            ts_slice = np.concatenate([ts_slice, pad])

        series_list.append(ts_slice.reshape(-1, 1))

    return np.stack(series_list)

In [ ]:
def prepare_parts(X_train, y_train, max_length: int = 1000, sample_idx=None):
    pairs_list = []
    sample_ids = y_train.index if sample_idx is None else y_train.iloc[sample_idx].index

    for ts_id in tqdm(sample_ids):
        ts = X_train.loc[ts_id]["value"].values.astype(np.float32).reshape(-1, 1)
        boundary = (X_train.loc[ts_id].period == 0).sum()
        start_left = max(0, boundary - max_length)
        end_right = min(len(ts), boundary + max_length)
        left_part = ts[start_left:boundary]
        right_part = ts[boundary:end_right]
        pairs_list.append([left_part, right_part])
    return pairs_list

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from ts2vec.utils import torch_pad_nan


class TS2VecEncoder:
    def __init__(self, model, batch_size):
        self.model = model
        self.batch_size = batch_size

    def encode(self, data, mode="default", **kwargs):
        if mode == "sliding":
            return self.encode_sliding(data, **kwargs)
        elif mode == "full_series":
            return self.encode_default(data, encoding_window="full_series", **kwargs)
        elif mode == "multiscale":
            return self.encode_default(data, encoding_window="multiscale", **kwargs)
        else:
            return self.encode_default(data, **kwargs)

    def encode_default(self, data, mask=None, encoding_window=None):
        assert data.ndim == 3
        self.model.eval()
        dataset = TensorDataset(torch.from_numpy(data).to(torch.float))
        loader = DataLoader(dataset, batch_size=self.batch_size)

        outputs = []
        with torch.no_grad():
            for batch in loader:
                x = batch[0]
                out = self._eval_with_pooling(x, mask, encoding_window=encoding_window)
                if encoding_window == "full_series":
                    out = out.squeeze(1)
                outputs.append(out)

        self.model.train()
        return torch.cat(outputs, dim=0).numpy()

    def encode_sliding(
        self, data, mask=None, encoding_window=None, causal=False, sliding_length=None, sliding_padding=0
    ):
        assert data.ndim == 3
        self.model.eval()
        batch_size = self.batch_size
        n_samples, ts_l, _ = data.shape

        dataset = TensorDataset(torch.from_numpy(data).to(torch.float))
        loader = DataLoader(dataset, batch_size=batch_size)

        outputs = []
        with torch.no_grad():
            for batch in loader:
                x = batch[0]
                reprs = []

                if n_samples < batch_size:
                    calc_buffer = []
                    calc_buffer_l = 0

                for i in range(0, ts_l, sliding_length):
                    l = i - sliding_padding
                    r = i + sliding_length + (0 if causal else sliding_padding)
                    x_sliding = torch_pad_nan(
                        x[:, max(l, 0) : min(r, ts_l)],
                        left=-l if l < 0 else 0,
                        right=r - ts_l if r > ts_l else 0,
                        dim=1,
                    )

                    if n_samples < batch_size:
                        if calc_buffer_l + n_samples > batch_size:
                            out = self._eval_with_pooling(
                                torch.cat(calc_buffer, dim=0),
                                mask,
                                slicing=slice(sliding_padding, sliding_padding + sliding_length),
                                encoding_window=encoding_window,
                            )
                            reprs += torch.split(out, n_samples)
                            calc_buffer = []
                            calc_buffer_l = 0

                        calc_buffer.append(x_sliding)
                        calc_buffer_l += n_samples
                    else:
                        out = self._eval_with_pooling(
                            x_sliding,
                            mask,
                            slicing=slice(sliding_padding, sliding_padding + sliding_length),
                            encoding_window=encoding_window,
                        )
                        reprs.append(out)

                if n_samples < batch_size and calc_buffer_l > 0:
                    out = self._eval_with_pooling(
                        torch.cat(calc_buffer, dim=0),
                        mask,
                        slicing=slice(sliding_padding, sliding_padding + sliding_length),
                        encoding_window=encoding_window,
                    )
                    reprs += torch.split(out, n_samples)

                out = torch.cat(reprs, dim=1)
                if encoding_window == "full_series":
                    out = F.max_pool1d(out.transpose(1, 2).contiguous(), kernel_size=out.size(1)).squeeze(1)

                outputs.append(out)

        self.model.train()
        return torch.cat(outputs, dim=0).numpy()

    def _eval_with_pooling(self, x, mask=None, slicing=None, encoding_window=None):
        device = next(self.model.parameters()).device
        out = self.model(x.to(device, non_blocking=True), mask)

        if encoding_window == "full_series":
            if slicing is not None:
                out = out[:, slicing]
            out = F.avg_pool1d(
                out.transpose(1, 2),
                kernel_size=out.size(1),
            ).transpose(1, 2)
            out = out.squeeze(1)

        elif isinstance(encoding_window, int):
            out = F.avg_pool1d(
                out.transpose(1, 2),
                kernel_size=encoding_window,
                stride=1,
                padding=encoding_window // 2,
            ).transpose(1, 2)
            if encoding_window % 2 == 0:
                out = out[:, :-1]
            if slicing is not None:
                out = out[:, slicing]

        elif encoding_window == "multiscale":
            p = 0
            reprs = []
            while (1 << p) + 1 < out.size(1):
                t_out = F.avg_pool1d(
                    out.transpose(1, 2),
                    kernel_size=(1 << (p + 1)) + 1,
                    stride=1,
                    padding=1 << p,
                ).transpose(1, 2)
                if slicing is not None:
                    t_out = t_out[:, slicing]
                reprs.append(t_out)
                p += 1
            out = torch.cat(reprs, dim=-1)

        else:
            if slicing is not None:
                out = out[:, slicing]

        return out.cpu()

In [ ]:
from ts2vec.utils import take_per_row
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from ts2vec.models.losses import hierarchical_contrastive_loss

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
# from sneddy_vec.encoder.dataset import CroppingDataset


class TS2VecTrainer:
    """The TS2Vec model"""

    def __init__(
        self,
        encoder,
        device="cuda",
        lr=0.001,
        temporal_unit=0,
        use_swa: bool = False
    ):
        super().__init__()
        self.device = device
        self.lr = lr
        self.temporal_unit = temporal_unit
        self.use_swa = use_swa
    
        self.encoder = encoder.to(self.device)
        if self.use_swa:
            self.averaged_encoder = torch.optim.swa_utils.AveragedModel(self._net)
            self.averaged_encoder.update_parameters(self._net)

        self.optimizer = torch.optim.AdamW(self.encoder.parameters(), lr=self.lr)

        self.current_epoch = 0
        self.n_iters = 0

    def fit_one_epoch(self, dataloader):
        self.encoder.train()  # включаем train mode
        for param in self.encoder.parameters():  # размораживаем encoder
            param.requires_grad = True
            
        cum_loss = 0
        n_epoch_iters = 0


        for first_crop, second_crop, crop_size in tqdm(dataloader):
            first_crop = first_crop.to(self.device)  # [B, L1, D]
            second_crop = second_crop.to(self.device)  # [B, L2, D]

            self.optimizer.zero_grad()

            out1 = self.encoder(first_crop)[:, -crop_size:]
            out2 = self.encoder(second_crop)[:, :crop_size]

            loss = hierarchical_contrastive_loss(out1, out2, temporal_unit=self.temporal_unit)
            loss.backward()
            self.optimizer.step()
            if self.use_swa:
                self.averaged_encoder.update_parameters(self._net)

            cum_loss += loss.item()
            n_epoch_iters += 1
            self.n_iters += 1

        cum_loss /= n_epoch_iters
        return cum_loss
        
    def validate(self, valid_data_left, valid_data_right, y_valid, batch_size:int=1):
        if self.use_swa:
            inference = TS2VecEncoder(self.averaged_encoder, batch_size=batch_size)
        else:
            inference = TS2VecEncoder(self.encoder, batch_size=batch_size)
        
        z_left = inference.encode_default(valid_data_left, encoding_window="full_series")   
        z_right = inference.encode_default(valid_data_right, encoding_window="full_series") 

        scores = 1 - np.array([
            cosine_similarity(zl.reshape(1, -1), zr.reshape(1, -1))[0, 0]
            for zl, zr in zip(z_left, z_right)
        ])

        auc = roc_auc_score(y_valid, scores)
        return auc

## Train TS2Vec

This is the only training cell you need for the showcase. It trains on one stratified fold and stores:

- `emb_trainer`
- `training_history`
- `valid_data_left`, `valid_data_right`
- `valid_idx`, `y_valid`
- `loss_log`


In [ ]:
from sklearn.model_selection import StratifiedKFold
from ts2vec.models import TSEncoder

kf = StratifiedKFold(shuffle=True, random_state=42)

EMB_BATCH_SIZE = 10
EMB_LR = 1e-3
N_EPOCHS = 15
LEFT_CONTEXT = 500
RIGHT_CONTEXT = 250
TRAIN_CONTEXT = 1000

for train_idx, valid_idx in kf.split(X=np.zeros(len(y_target)), y=y_target):
    train_data = prepare_positive_train_data(
        X_train,
        y_train,
        left_length=-TRAIN_CONTEXT,
        right_length=0,
        sample_idx=train_idx,
    )
    valid_data_left = prepare_positive_train_data(
        X_train,
        y_train,
        left_length=-LEFT_CONTEXT,
        right_length=0,
        sample_idx=valid_idx,
    )
    valid_data_right = prepare_positive_train_data(
        X_train,
        y_train,
        left_length=0,
        right_length=RIGHT_CONTEXT,
        sample_idx=valid_idx,
    )
    y_valid = y_target[valid_idx]

    encoder = TSEncoder(
        input_dims=1,
        output_dims=64,
        hidden_dims=128,
        depth=10,
    )
    dataset_emb = CroppingDataset(
        data_list=train_data,
        seq_length=600,
        temporal_unit=5,
        diff_proba=0.3,
    )
    train_loader_emb = DataLoader(
        dataset_emb,
        batch_size=EMB_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        collate_fn=dataset_emb.collate_fn,
    )
    emb_trainer = TS2VecTrainer(
        encoder,
        lr=EMB_LR,
        temporal_unit=5,
        use_swa=False,
        device="cuda",
    )

    history_rows = []
    best_emb_score = 0.0
    best_score_weights = None

    for epoch in range(1, N_EPOCHS + 1):
        emb_loss = emb_trainer.fit_one_epoch(train_loader_emb)
        emb_val_score = emb_trainer.validate(valid_data_left, valid_data_right, y_valid, batch_size=10)
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": float(emb_loss),
                "val_auc": float(emb_val_score),
            }
        )
        print(f"[epoch {epoch:02d}] loss={emb_loss:.4f} | val_auc={emb_val_score:.4f}")
        if emb_val_score > best_emb_score:
            best_emb_score = float(emb_val_score)
            best_score_weights = emb_trainer.encoder.state_dict()

    training_history = pd.DataFrame(history_rows)
    loss_log = training_history["train_loss"].tolist()
    break

training_history.tail()

## Score Structural Breaks

After the 15 epochs, compute cosine mismatch scores once. The diagnostics block below can be run immediately after this cell.

In [ ]:
inference = TS2VecEncoder(emb_trainer.encoder, batch_size=64)

z_left = inference.encode_default(valid_data_left, encoding_window="full_series")
z_right = inference.encode_default(valid_data_right, encoding_window="full_series")
cosine_scores = 1 - np.array([
    cosine_similarity(zl.reshape(1, -1), zr.reshape(1, -1))[0, 0]
    for zl, zr in zip(z_left, z_right)
])

valid_data_pairs = prepare_parts(X_train, y_train, sample_idx=valid_idx)
score_auc = roc_auc_score(y_valid, cosine_scores)
print(f"Validation cosine ROC AUC: {score_auc:.4f}")

## Presentation Diagnostics For Structural Break Prediction

Run the cells below after the scoring cell. They produce the full minimal showcase:

- training dynamics
- ROC and score separation
- embedding shift map across the candidate breakpoint
- representative no-break and break examples


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

sns.set_theme(style="whitegrid", context="talk")
PLOT_COLORS = {
    "no_break": "#4C78A8",
    "break": "#E45756",
    "left": "#4C78A8",
    "right": "#F58518",
    "accent": "#54A24B",
    "neutral": "#B279A2",
}


def resolve_ts2vec_backbone():
    if "emb_trainer" in globals() and hasattr(emb_trainer, "encoder"):
        return emb_trainer.encoder
    if "trainer" in globals():
        for attr in ("encoder", "net", "_net"):
            candidate = getattr(trainer, attr, None)
            if candidate is not None:
                return candidate
        if hasattr(trainer, "encode"):
            return trainer
    raise RuntimeError(
        "Could not find a trained TS2Vec object. Run one of the structural-break training cells first."
    )


def encode_validation_embeddings(valid_data_left, valid_data_right, batch_size=64):
    backbone = resolve_ts2vec_backbone()

    if hasattr(backbone, "encode") and not isinstance(backbone, torch.nn.Module):
        z_left = backbone.encode(valid_data_left, encoding_window="full_series", batch_size=batch_size)
        z_right = backbone.encode(valid_data_right, encoding_window="full_series", batch_size=batch_size)
    else:
        inference = TS2VecEncoder(backbone, batch_size=batch_size)
        z_left = inference.encode_default(valid_data_left, encoding_window="full_series")
        z_right = inference.encode_default(valid_data_right, encoding_window="full_series")

    z_left = np.asarray(z_left)
    z_right = np.asarray(z_right)
    if z_left.ndim == 3:
        z_left = z_left.squeeze(1)
    if z_right.ndim == 3:
        z_right = z_right.squeeze(1)
    return z_left, z_right


def build_score_frame(cosine_scores, y_valid, valid_data_pairs):
    y_valid = np.asarray(y_valid).reshape(-1)
    cosine_scores = np.asarray(cosine_scores).reshape(-1)

    rows = []
    for sample_id, ((left, right), label, score) in enumerate(zip(valid_data_pairs, y_valid, cosine_scores)):
        left_series = np.asarray(left).reshape(-1)
        right_series = np.asarray(right).reshape(-1)
        rows.append(
            {
                "sample_id": sample_id,
                "label": int(label),
                "label_name": "break" if int(label) == 1 else "no break",
                "score": float(score),
                "left_len": int(left_series.shape[0]),
                "right_len": int(right_series.shape[0]),
                "left_mean": float(np.nanmean(left_series)),
                "right_mean": float(np.nanmean(right_series)),
                "mean_shift": float(np.nanmean(right_series) - np.nanmean(left_series)),
                "boundary_jump": float(right_series[0] - left_series[-1]) if len(left_series) and len(right_series) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def plot_training_history(training_history):
    history = pd.DataFrame(training_history).copy()
    epochs = history["epoch"].to_numpy()
    best_idx = int(history["val_auc"].idxmax())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    axes[0].plot(epochs, history["train_loss"], color=PLOT_COLORS["break"], linewidth=2.5, marker="o", markersize=4)
    axes[0].scatter(history.loc[best_idx, "epoch"], history.loc[best_idx, "train_loss"], s=140, color=PLOT_COLORS["accent"], zorder=3)
    axes[0].set_title("TS2Vec training loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Contrastive loss")

    axes[1].plot(epochs, history["val_auc"], color=PLOT_COLORS["no_break"], linewidth=2.5, marker="o", markersize=4)
    axes[1].scatter(history.loc[best_idx, "epoch"], history.loc[best_idx, "val_auc"], s=140, color=PLOT_COLORS["accent"], zorder=3)
    axes[1].annotate(
        f"best epoch = {int(history.loc[best_idx, 'epoch'])}\nAUC = {history.loc[best_idx, 'val_auc']:.3f}",
        xy=(history.loc[best_idx, "epoch"], history.loc[best_idx, "val_auc"]),
        xytext=(12, -28),
        textcoords="offset points",
        fontsize=11,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.9},
    )
    axes[1].set_title("Validation cosine ROC AUC")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("ROC AUC")

    fig.suptitle("TS2Vec Training Dynamics On The Structural-Break Dataset", y=1.02, fontsize=18)
    plt.tight_layout()
    plt.show()


def plot_score_diagnostics(score_df):
    labels = score_df["label"].to_numpy()
    scores = score_df["score"].to_numpy()

    fpr, tpr, thresholds = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    avg_precision = average_precision_score(labels, scores)
    best_idx = int(np.argmax(tpr - fpr))
    best_threshold = thresholds[best_idx]

    fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

    axes[0].plot(fpr, tpr, color=PLOT_COLORS["break"], linewidth=2.5, label=f"ROC AUC = {roc_auc:.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="#666666", linewidth=1)
    axes[0].scatter(fpr[best_idx], tpr[best_idx], color=PLOT_COLORS["accent"], s=90, zorder=3)
    axes[0].set_title("Validation ROC")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].legend(loc="lower right")
    axes[0].text(
        0.04,
        0.1,
        f"Average precision = {avg_precision:.3f}\nBest threshold = {best_threshold:.3f}",
        transform=axes[0].transAxes,
        fontsize=11,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.9},
    )

    sns.histplot(
        data=score_df,
        x="score",
        hue="label_name",
        bins=40,
        element="step",
        stat="density",
        common_norm=False,
        alpha=0.25,
        palette={"no break": PLOT_COLORS["no_break"], "break": PLOT_COLORS["break"]},
        ax=axes[1],
    )
    axes[1].axvline(best_threshold, color=PLOT_COLORS["accent"], linestyle="--", linewidth=2)
    axes[1].set_title("Cosine-Mismatch Score Distribution")
    axes[1].set_xlabel("1 - cosine similarity(left, right)")
    axes[1].set_ylabel("Density")

    ranked = score_df.sort_values("score").reset_index(drop=True)
    axes[2].scatter(
        ranked.index,
        ranked["score"],
        c=ranked["label"].map({0: PLOT_COLORS["no_break"], 1: PLOT_COLORS["break"]}),
        s=16,
        alpha=0.8,
    )
    axes[2].axhline(best_threshold, color=PLOT_COLORS["accent"], linestyle="--", linewidth=2)
    axes[2].set_title("Ranked Validation Scores")
    axes[2].set_xlabel("Validation sample rank")
    axes[2].set_ylabel("Cosine mismatch score")

    handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PLOT_COLORS["no_break"], markersize=8, label="no break"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PLOT_COLORS["break"], markersize=8, label="break"),
    ]
    axes[2].legend(handles=handles, loc="upper left")

    plt.tight_layout()
    plt.show()


def balanced_sample_indices(labels, max_per_class=175, random_state=42):
    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels)
    sampled = []
    for label in np.unique(labels):
        label_idx = np.flatnonzero(labels == label)
        take = min(max_per_class, len(label_idx))
        sampled.extend(rng.choice(label_idx, size=take, replace=False).tolist())
    sampled = np.array(sorted(sampled))
    return sampled


def plot_embedding_shift_map(z_left, z_right, y_valid, max_per_class=175):
    y_valid = np.asarray(y_valid).reshape(-1)
    sample_idx = balanced_sample_indices(y_valid, max_per_class=max_per_class)

    sampled_left = z_left[sample_idx]
    sampled_right = z_right[sample_idx]
    sampled_labels = y_valid[sample_idx]

    projector = PCA(n_components=2, random_state=42)
    projected = projector.fit_transform(np.vstack([sampled_left, sampled_right]))
    left_proj = projected[: len(sample_idx)]
    right_proj = projected[len(sample_idx) :]

    fig, ax = plt.subplots(figsize=(11, 9))
    class_meta = {
        0: {"color": PLOT_COLORS["no_break"], "name": "no break"},
        1: {"color": PLOT_COLORS["break"], "name": "break"},
    }

    for label, meta in class_meta.items():
        mask = sampled_labels == label
        for start, end in zip(left_proj[mask], right_proj[mask]):
            ax.plot(
                [start[0], end[0]],
                [start[1], end[1]],
                color=meta["color"],
                alpha=0.08,
                linewidth=1.2,
            )
        ax.scatter(left_proj[mask, 0], left_proj[mask, 1], color=meta["color"], alpha=0.20, s=24, marker="o")
        ax.scatter(right_proj[mask, 0], right_proj[mask, 1], color=meta["color"], alpha=0.85, s=28, marker="x")

        centroid_left = left_proj[mask].mean(axis=0)
        centroid_right = right_proj[mask].mean(axis=0)
        ax.scatter(*centroid_left, color=meta["color"], s=240, marker="o", edgecolor="white", linewidth=1.5)
        ax.scatter(*centroid_right, color=meta["color"], s=260, marker="X", edgecolor="white", linewidth=1.5)
        ax.annotate(
            "",
            xy=centroid_right,
            xytext=centroid_left,
            arrowprops={"arrowstyle": "-|>", "lw": 3, "color": meta["color"]},
        )

    ax.set_title("TS2Vec Embedding Shift Across The Candidate Breakpoint")
    ax.set_xlabel(f"PC1 ({projector.explained_variance_ratio_[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({projector.explained_variance_ratio_[1]:.1%} variance)")

    legend_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PLOT_COLORS["neutral"], markeredgecolor="white", markersize=10, label="left context embedding"),
        Line2D([0], [0], marker="x", color=PLOT_COLORS["neutral"], markersize=10, linewidth=0, label="right context embedding"),
        Line2D([0], [0], color=PLOT_COLORS["no_break"], lw=3, label="no-break pair"),
        Line2D([0], [0], color=PLOT_COLORS["break"], lw=3, label="break pair"),
    ]
    ax.legend(handles=legend_handles, loc="best", frameon=True)
    plt.tight_layout()
    plt.show()


def plot_boundary_gallery(valid_data_pairs, score_df, n_per_class=3):
    no_break_examples = score_df.query("label == 0").nsmallest(n_per_class, "score")
    break_examples = score_df.query("label == 1").nlargest(n_per_class, "score")
    groups = [
        ("No-break pairs with the smallest embedding mismatch", no_break_examples),
        ("Break pairs with the largest embedding mismatch", break_examples),
    ]

    fig, axes = plt.subplots(2, n_per_class, figsize=(5.2 * n_per_class, 8), sharey=False)
    if n_per_class == 1:
        axes = np.array(axes).reshape(2, 1)

    for row_idx, (row_title, subset) in enumerate(groups):
        subset = subset.reset_index(drop=True)
        for col_idx in range(n_per_class):
            ax = axes[row_idx, col_idx]
            if col_idx >= len(subset):
                ax.axis("off")
                continue

            sample = subset.iloc[col_idx]
            left, right = valid_data_pairs[int(sample.sample_id)]
            left = np.asarray(left).reshape(-1)
            right = np.asarray(right).reshape(-1)
            x_left = np.arange(-len(left), 0)
            x_right = np.arange(0, len(right))

            ax.plot(x_left, left, color=PLOT_COLORS["left"], linewidth=2.0)
            ax.plot(x_right, right, color=PLOT_COLORS["right"], linewidth=2.0)
            ax.axvline(0, color="#222222", linestyle="--", linewidth=1.2)
            ax.axvspan(-3, 3, color="#222222", alpha=0.06)
            ax.set_title(f"sample #{int(sample.sample_id)} | score = {sample.score:.3f}")
            ax.set_xlabel("Time relative to boundary")
            if col_idx == 0:
                ax.set_ylabel(row_title)

    fig.suptitle("Representative Structural-Break Pairs", y=1.02, fontsize=18)
    plt.tight_layout()
    plt.show()


In [ ]:
if "training_history" in globals() and len(training_history):
    plot_training_history(training_history)
elif "loss_log" in globals() and len(loss_log):
    plot_training_history(pd.DataFrame({"epoch": np.arange(1, len(loss_log) + 1), "train_loss": loss_log, "val_auc": np.nan}))
else:
    print("No training history found; run the training cell first.")

In [ ]:
score_df = build_score_frame(cosine_scores, y_valid, valid_data_pairs)
score_df.groupby("label_name")["score"].describe().round(3)

In [ ]:
plot_score_diagnostics(score_df)

In [ ]:
if "z_left" not in globals() or "z_right" not in globals():
    z_left, z_right = encode_validation_embeddings(valid_data_left, valid_data_right, batch_size=64)
plot_embedding_shift_map(z_left, z_right, y_valid, max_per_class=175)

In [ ]:
plot_boundary_gallery(valid_data_pairs, score_df, n_per_class=3)